In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
import pickle
import boto3
import json
import re
import time
import unicodedata
from io import BytesIO
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from xgboost import XGBRegressor

# Control de promoción de modelo
AUTO_PROMOTE_IF_BETTER = True
FORCE_PROMOTE_TO_CHAMPION = False
USE_HISTORY_FEATURES = True

# =============================================================
# 1. CARGAR DATOS — Silver deduped (preferido) + historial Silver
# =============================================================
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
        "bucket_name": "bronce-scrap-date",
    }
except Exception:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)

BUCKET = config["bucket_name"]
S3_OPTS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

TRAINING_SNAPSHOT_PATH = config.get("training_snapshot_path") or f"s3a://{BUCKET}/silver/master_deduped/"
HISTORY_DATA_PATH = config.get("history_data_path") or f"s3a://{BUCKET}/silver/master_inmuebles/"
LEGACY_GOLD_PATH = config.get("legacy_gold_path") or f"s3a://{BUCKET}/gold/app_inmuebles/"
print(f"📦 Bucket configurado: {BUCKET}")
print(f"📍 Snapshot ML: {TRAINING_SNAPSHOT_PATH}")
print(f"📍 Historial ML: {HISTORY_DATA_PATH}")
print(f"📍 Fallback Gold: {LEGACY_GOLD_PATH}")

def summarize_spark_exception(exc):
    message = str(exc)
    message = re.sub(r"\s+", " ", message).strip()
    for token in ["JVM stacktrace:", "Caused by:", "Trace ID:"]:
        if token in message:
            message = message.split(token, 1)[0].strip()
    if "403 Forbidden" in message or "UNAUTHORIZED_ACCESS" in message:
        return "acceso denegado (403) al dataset de historial en S3"
    return message[:280] + ("..." if len(message) > 280 else "")

def load_spark_path(path, formats):
    last_error = None
    for fmt in formats:
        try:
            reader = spark.read.format(fmt)
            for k, v in S3_OPTS.items():
                reader = reader.option(k, v)
            return reader.load(path), fmt
        except Exception as exc:
            last_error = exc
    raise last_error

try:
    ruta_deduped = TRAINING_SNAPSHOT_PATH
    df_spark, deduped_format = load_spark_path(ruta_deduped, ["delta"])
    print(f"📊 Leyendo Silver Deduped ({deduped_format.upper()}): {ruta_deduped}")
except Exception:
    ruta_gold = LEGACY_GOLD_PATH
    df_spark, gold_format = load_spark_path(ruta_gold, ["parquet", "delta"])
    print(f"📊 Fallback → Gold legacy ({gold_format.upper()}): {ruta_gold}")

print(f"   Columnas deduped: {df_spark.columns}")

df_history_spark = None
history_read_enabled = USE_HISTORY_FEATURES
if history_read_enabled:
    ruta_history = HISTORY_DATA_PATH
    try:
        df_history_spark, history_format = load_spark_path(ruta_history, ["delta", "parquet"])
        print(f"🕒 Historial disponible ({history_format.upper()}): {ruta_history}")
        print(f"   Columnas historial: {df_history_spark.columns}")
    except Exception as e:
        history_read_enabled = False
        print(f"⚠️ Historial deshabilitado en esta corrida: {summarize_spark_exception(e)}")
else:
    print("ℹ️ Features históricas deshabilitadas por configuración (USE_HISTORY_FEATURES=False).")

# =============================================================
# 2. PREPARAR FEATURES BASE
# =============================================================
available = set(df_spark.columns)
history_available = set(df_history_spark.columns) if df_history_spark is not None else set()

base_numeric_candidates = [
    "area_m2", "habitaciones", "banos", "garajes",
    "num_portales", "dispersion_pct_grupo", "precio_desviacion_grupo_pct",
    "data_completeness",
]
cols_numeric_base = [c for c in base_numeric_candidates if c in available]
cols_categorical = [
    c for c in ["tipo_inmueble", "estado_inmueble", "fuente", "city_token"]
    if c in available
]
cols_text = [c for c in ["ubicacion_norm", "ubicacion_raw", "titulo"] if c in available]
aux_cols = [
    c for c in ["property_group_id", "id_original", "fecha_extraccion", "url", "batch_id", "source_file"]
    if c in available
]
all_cols = list(dict.fromkeys(["precio_num"] + cols_numeric_base + cols_categorical + cols_text + aux_cols))
print(f"📋 Features base: numeric={cols_numeric_base}, cat={cols_categorical}, text={cols_text}")

df = df_spark.select(*all_cols).toPandas()
print(f"   Total registros: {len(df):,}")

for col in cols_numeric_base + ["precio_num"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "fecha_extraccion" in df.columns:
    df["fecha_extraccion"] = pd.to_datetime(df["fecha_extraccion"], errors="coerce")

string_cols = list(dict.fromkeys(cols_categorical + cols_text + ["property_group_id", "id_original", "url", "batch_id", "source_file"]))
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

if "city_token" not in df.columns:
    df["city_token"] = "otra_ciudad"
if "tipo_inmueble" not in df.columns:
    df["tipo_inmueble"] = "otro"
if "fuente" not in df.columns:
    df["fuente"] = "desconocido"

# city_token se normaliza e infiere más adelante usando ubicacion_limpia.

tipos_validos = df["tipo_inmueble"].value_counts()
tipos_validos = tipos_validos[tipos_validos >= 40].index
df["tipo_inmueble"] = df["tipo_inmueble"].where(df["tipo_inmueble"].isin(tipos_validos), other="otro")

text_parts = [df[col].fillna("") for col in cols_text]
if text_parts:
    df["texto_completo"] = text_parts[0]
    for part in text_parts[1:]:
        df["texto_completo"] = df["texto_completo"] + " " + part
else:
    df["texto_completo"] = ""

def normalize_text(value):
    text = "" if pd.isna(value) else str(value)
    text = text.lower().strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\|.*", "", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_url(value):
    text = "" if pd.isna(value) else str(value).strip().lower()
    text = re.sub(r"\?.*$", "", text)
    text = re.sub(r"/$", "", text)
    return text

def build_listing_key(frame):
    fuente_part = frame["fuente"].fillna("desconocido").astype(str) if "fuente" in frame.columns else pd.Series("desconocido", index=frame.index)
    if "id_original" in frame.columns:
        id_part = frame["id_original"].fillna("").astype(str).str.strip()
    else:
        id_part = pd.Series("", index=frame.index)
    if "url" in frame.columns:
        url_part = frame["url"].fillna("").astype(str).apply(normalize_url)
    else:
        url_part = pd.Series("", index=frame.index)
    if "property_group_id" in frame.columns:
        group_part = frame["property_group_id"].fillna("").astype(str).str.strip()
    else:
        group_part = pd.Series("", index=frame.index)

    key = np.where(
        id_part.ne(""),
        fuente_part + "__id__" + id_part,
        np.where(
            url_part.ne(""),
            fuente_part + "__url__" + url_part,
            np.where(group_part.ne(""), "group__" + group_part, "")
        ),
    )
    return pd.Series(key, index=frame.index)

COMUNA_KEYWORDS_BY_CITY = {
    "medellin": {
        "poblado": [
            "poblado", "provenza", "manila", "castropol", "lalinde",
            "aguacatala", "patio bonito", "los balsos", "el tesoro",
            "santa maria de los angeles",
        ],
        "laureles_estadio": [
            "laureles", "estadio", "la america", "america", "floresta",
            "calasanz", "suramericana", "velodromo", "los colores",
            "simon bolivar", "carlos e restrepo",
        ],
        "belen": [
            "belen", "fatima", "rosales", "granada", "loma de los bernal",
            "nogal", "aliadas", "mira valle",
        ],
        "robledo": [
            "robledo", "pajarito", "pilarica", "cordoba", "villa flora",
            "aures", "altamira",
        ],
        "castilla_doce_octubre": [
            "castilla", "doce de octubre", "pedregal", "tejelo", "girardot",
            "florencia", "boyaca",
        ],
        "aranjuez_manrique": [
            "aranjuez", "manrique", "berlin", "campo valdes", "palos verdes",
            "miraflores",
        ],
        "popular_santa_cruz": [
            "popular", "santa cruz", "andalucia", "moscu", "villa guadalupe",
        ],
        "buenos_aires": [
            "buenos aires", "bombona", "la milagrosa", "barcelona", "gerona",
        ],
        "guayabal": [
            "guayabal", "trinidad", "cristo rey", "santa fe",
        ],
        "centro": [
            "candelaria", "centro", "prado", "sevilla", "san diego",
        ],
    },
    "bogota": {
        "usaquen": ["usaquen", "santa barbara", "cedritos", "bella suiza", "country"],
        "chapinero": ["chapinero", "rosales", "nogal", "chico", "castellana", "quinta camacho"],
        "suba": ["suba", "niza", "colina", "gratamira", "mazuren"],
        "teusaquillo_barrios_unidos": ["teusaquillo", "salitre", "parque simon bolivar", "barrios unidos"],
        "kennedy_fontibon": ["kennedy", "fontibon", "hayuelos", "modelia", "castilla"],
    },
}

ubicacion_base_col = "ubicacion_norm" if "ubicacion_norm" in df.columns else ("ubicacion_raw" if "ubicacion_raw" in df.columns else None)
if ubicacion_base_col is not None:
    df["ubicacion_limpia"] = df[ubicacion_base_col].apply(normalize_text)
else:
    df["ubicacion_limpia"] = ""

df["city_token"] = df["city_token"].apply(normalize_text)
df["city_token"] = df["city_token"].replace("", "otra_ciudad")

CITY_ALIAS_TO_TOKEN = {
    "bogota": ["bogota", "bogota dc", "bogota d c", "bogota distrito capital"],
    "medellin": ["medellin", "medallo"],
    "cali": ["cali", "santiago de cali"],
    "barranquilla": ["barranquilla", "bquilla"],
    "cartagena": ["cartagena", "cartagena de indias"],
    "bucaramanga": ["bucaramanga"],
    "pereira": ["pereira"],
    "manizales": ["manizales"],
    "cucuta": ["cucuta", "san jose de cucuta"],
    "santa marta": ["santa marta"],
}

def canonicalize_city_token(raw_city_token):
    city_value = normalize_text(raw_city_token)
    if not city_value or city_value == "otra_ciudad":
        return "otra_ciudad"
    for city_token, aliases in CITY_ALIAS_TO_TOKEN.items():
        if city_value == city_token or any(alias in city_value for alias in aliases):
            return city_token
    return city_value

def infer_city_from_location(ubicacion_limpia):
    location = normalize_text(ubicacion_limpia)
    if not location:
        return None

    for city_token, aliases in CITY_ALIAS_TO_TOKEN.items():
        if any(alias in location for alias in aliases):
            return city_token

    ambiguous_zone_tokens = {"castilla", "granada", "centro", "prado", "florencia"}
    for city_token, comuna_map in COMUNA_KEYWORDS_BY_CITY.items():
        for keywords in comuna_map.values():
            if any((keyword in location) and (keyword not in ambiguous_zone_tokens) for keyword in keywords):
                return city_token

    return None

df["city_token_raw"] = df["city_token"]
df["city_token"] = df["city_token"].apply(canonicalize_city_token)
conteos_raw_city = df["city_token"].value_counts()
ciudades_validas_pre = set(conteos_raw_city[conteos_raw_city >= 80].index)
known_city_tokens = set(CITY_ALIAS_TO_TOKEN.keys())
mask_city_needs_inference = df["city_token"].eq("otra_ciudad") | ~df["city_token"].isin(ciudades_validas_pre.union(known_city_tokens))
city_inferred = df.loc[mask_city_needs_inference, "ubicacion_limpia"].apply(infer_city_from_location)
df.loc[mask_city_needs_inference, "city_token"] = city_inferred.fillna(df.loc[mask_city_needs_inference, "city_token"])
city_recovered = int((mask_city_needs_inference & df["city_token"].ne(df["city_token_raw"]) & df["city_token"].ne("otra_ciudad")).sum())

conteos = df["city_token"].value_counts()
ciudades_validas = conteos[conteos >= 80].index
df["city_token"] = df["city_token"].where(df["city_token"].isin(ciudades_validas), other="otra_ciudad")
print(f"   city_token: {df['city_token'].nunique()} categorías después de normalizar+inferir+colapso (umbral=80)")
print(f"   city_token reasignados por normalización/ubicación antes del colapso: {city_recovered:,}")

def assign_comuna(city_token, ubicacion_limpia):
    city = normalize_text(city_token)
    location = normalize_text(ubicacion_limpia)
    city_map = COMUNA_KEYWORDS_BY_CITY.get(city, {})
    for comuna_name, keywords in city_map.items():
        if any(keyword in location for keyword in keywords):
            return comuna_name
    return "comuna_otra"

df["comuna_mercado"] = [
    assign_comuna(city, location)
    for city, location in zip(df["city_token"], df["ubicacion_limpia"])
]

df["listing_history_key"] = build_listing_key(df)
comunas_validas = df["comuna_mercado"].value_counts()
comunas_validas = comunas_validas[comunas_validas >= 25].index
df["comuna_mercado"] = df["comuna_mercado"].where(df["comuna_mercado"].isin(comunas_validas), other="comuna_otra")
print(f"   comuna_mercado: {df['comuna_mercado'].nunique()} categorías después de colapso (umbral=25)")

# =============================================================
# 2.1 FEATURES EXPLÍCITAS: CONSENSO, AMENITIES, HISTORIAL
# =============================================================
completeness_fill = df["data_completeness"].median() if "data_completeness" in df.columns and df["data_completeness"].notna().any() else 0.5
num_portales_norm = df["num_portales"].fillna(1).clip(lower=1, upper=4) if "num_portales" in df.columns else pd.Series(1.0, index=df.index)
dispersion = df["dispersion_pct_grupo"].fillna(50).clip(lower=0, upper=100) if "dispersion_pct_grupo" in df.columns else pd.Series(50.0, index=df.index)
desv = df["precio_desviacion_grupo_pct"].fillna(50).clip(lower=0, upper=100) if "precio_desviacion_grupo_pct" in df.columns else pd.Series(50.0, index=df.index)
completeness = df["data_completeness"].fillna(completeness_fill) if "data_completeness" in df.columns else pd.Series(completeness_fill, index=df.index)
if completeness.max(skipna=True) > 1.5:
    completeness = completeness / 100.0
completeness = completeness.clip(lower=0.0, upper=1.0)

df["score_consenso_cross_portal"] = (
    np.clip((num_portales_norm - 1.0) / 3.0, 0, 1) * 35.0
    + (1.0 - dispersion / 100.0) * 30.0
    + (1.0 - desv / 100.0) * 20.0
    + completeness * 15.0
).clip(lower=0.0, upper=100.0)

AMENITY_PATTERNS = {
    "amenity_balcon": ["balcon", "balcony"],
    "amenity_terraza": ["terraza", "roof garden", "patio"],
    "amenity_ascensor": ["ascensor", "elevador"],
    "amenity_deposito": ["deposito", "bodega", "locker", "cuarto util"],
    "amenity_estudio": ["estudio", "office", "biblioteca"],
    "amenity_remodelado": ["remodelado", "renovado", "reformado", "para estrenar"],
    "amenity_gimnasio": ["gimnasio", "gym"],
    "amenity_piscina": ["piscina", "pool", "jacuzzi"],
    "amenity_conjunto": ["conjunto", "unidad cerrada", "urbanizacion cerrada"],
    "amenity_vigilancia": ["vigilancia", "porter", "porteria", "seguridad"],
    "amenity_vista": ["vista", "vista panoramica", "vista al mar", "vista verde"],
    "amenity_penthouse": ["penthouse", "ph"],
    "amenity_duplex": ["duplex", "triplex"],
    "amenity_amoblado": ["amoblado", "amoblada", "full amoblado"],
    "amenity_lujo": ["lujo", "exclusivo", "premium", "alta valorizacion"],
}
text_signal = (df["texto_completo"].fillna("") + " " + df["ubicacion_limpia"].fillna(" ")).apply(normalize_text)
amenity_feature_cols = []
for feature_name, keywords in AMENITY_PATTERNS.items():
    pattern = r"\\b(?:" + "|".join(re.escape(keyword) for keyword in keywords) + r")\\b"
    df[feature_name] = text_signal.str.contains(pattern, regex=True, na=False).astype(float)
    amenity_feature_cols.append(feature_name)

df["amenities_count"] = df[amenity_feature_cols].sum(axis=1)
df["amenity_lujo_score"] = df[[
    "amenity_balcon", "amenity_terraza", "amenity_ascensor", "amenity_gimnasio",
    "amenity_piscina", "amenity_vista", "amenity_lujo",
]].sum(axis=1)

history_feature_cols = [
    "dias_en_mercado", "republicaciones_count", "num_repricings",
    "descuento_desde_inicial_hist_pct", "dias_desde_ultimo_repricing",
    "n_observaciones_hist", "n_batches_hist", "n_urls_hist",
]
history_features = pd.DataFrame(columns=["listing_history_key"] + history_feature_cols)
history_feature_coverage = {col: 0.0 for col in history_feature_cols}

if USE_HISTORY_FEATURES and df_history_spark is not None:
    history_cols = [
        c for c in ["fuente", "id_original", "url", "property_group_id", "fecha_extraccion", "precio_num", "batch_id", "source_file"]
        if c in history_available
    ]
    try:
        history_t0 = time.time()
        history_df = df_history_spark.select(*history_cols).toPandas()
        print(f"🕒 Registros historial cargados: {len(history_df):,}")
        for col in ["fuente", "id_original", "url", "property_group_id", "batch_id", "source_file"]:
            if col in history_df.columns:
                history_df[col] = history_df[col].fillna("").astype(str)
        if "fecha_extraccion" in history_df.columns:
            history_df["fecha_extraccion"] = pd.to_datetime(history_df["fecha_extraccion"], errors="coerce")
        if "precio_num" in history_df.columns:
            history_df["precio_num"] = pd.to_numeric(history_df["precio_num"], errors="coerce")

        history_df["listing_history_key"] = build_listing_key(history_df)
        history_df = history_df[
            history_df["listing_history_key"].ne("") &
            history_df["fecha_extraccion"].notna()
        ].copy()
        history_df = history_df.sort_values(["listing_history_key", "fecha_extraccion"], kind="stable")

        history_base = (
            history_df.groupby("listing_history_key")
            .agg(
                first_date=("fecha_extraccion", "min"),
                last_date=("fecha_extraccion", "max"),
                n_observaciones_hist=("fecha_extraccion", "size"),
            )
            .reset_index()
        )

        history_df["gap_days"] = history_df.groupby("listing_history_key")["fecha_extraccion"].diff().dt.days
        history_df["gap_gt_45"] = history_df["gap_days"].gt(45).astype(int)
        gap_stats = (
            history_df.groupby("listing_history_key", as_index=False)["gap_gt_45"]
            .sum()
            .rename(columns={"gap_gt_45": "gap_count"})
        )

        url_stats = pd.DataFrame({"listing_history_key": history_base["listing_history_key"]})
        if "url" in history_df.columns:
            url_stats = (
                history_df.assign(url_non_empty=history_df["url"].where(history_df["url"].ne("")))
                .groupby("listing_history_key")["url_non_empty"]
                .nunique(dropna=True)
                .reset_index(name="n_urls_hist")
            )

        batch_stats = pd.DataFrame({"listing_history_key": history_base["listing_history_key"]})
        if "batch_id" in history_df.columns:
            batch_stats = (
                history_df.assign(batch_non_empty=history_df["batch_id"].where(history_df["batch_id"].ne("")))
                .groupby("listing_history_key")["batch_non_empty"]
                .nunique(dropna=True)
                .reset_index(name="n_batches_hist")
            )

        price_events = history_df.loc[history_df["precio_num"].notna(), ["listing_history_key", "fecha_extraccion", "precio_num"]].copy()
        price_stats = pd.DataFrame(columns=["listing_history_key", "num_repricings", "first_price", "prev_price", "last_reprice_date"])
        if not price_events.empty:
            price_events = price_events.sort_values(["listing_history_key", "fecha_extraccion", "precio_num"], kind="stable")
            prev_price_series = price_events.groupby("listing_history_key")["precio_num"].shift()
            price_events["price_changed"] = price_events["precio_num"].ne(prev_price_series)
            price_stats = (
                price_events.groupby("listing_history_key")
                .agg(
                    changed_events=("price_changed", "sum"),
                    first_price=("precio_num", "first"),
                )
                .reset_index()
            )
            price_stats["num_repricings"] = (price_stats["changed_events"] - 1).clip(lower=0)
            last_reprice = (
                price_events.loc[price_events["price_changed"], ["listing_history_key", "fecha_extraccion"]]
                .groupby("listing_history_key", as_index=False)["fecha_extraccion"]
                .max()
                .rename(columns={"fecha_extraccion": "last_reprice_date"})
            )
            price_events["price_order"] = price_events.groupby("listing_history_key").cumcount()
            price_events["price_count"] = price_events.groupby("listing_history_key")["precio_num"].transform("size")
            prev_price = price_events.loc[
                (price_events["price_count"] == 1) | (price_events["price_order"] == price_events["price_count"] - 2),
                ["listing_history_key", "precio_num"],
            ].rename(columns={"precio_num": "prev_price"})
            price_stats = price_stats.merge(last_reprice, on="listing_history_key", how="left")
            price_stats = price_stats.merge(prev_price, on="listing_history_key", how="left")
            price_stats = price_stats.drop(columns=["changed_events"], errors="ignore")
            price_stats["prev_price"] = price_stats["prev_price"].fillna(price_stats["first_price"])

        history_features = history_base.merge(gap_stats, on="listing_history_key", how="left")
        history_features = history_features.merge(url_stats, on="listing_history_key", how="left")
        history_features = history_features.merge(batch_stats, on="listing_history_key", how="left")
        history_features = history_features.merge(price_stats, on="listing_history_key", how="left")

        history_features["gap_count"] = history_features["gap_count"].fillna(0)
        history_features["n_urls_hist"] = history_features.get("n_urls_hist", 0)
        history_features["n_batches_hist"] = history_features.get("n_batches_hist", 0)
        history_features["num_repricings"] = history_features.get("num_repricings", 0).fillna(0)
        history_features["republicaciones_count"] = np.maximum(history_features["n_urls_hist"].fillna(0) - 1, history_features["gap_count"]).clip(lower=0)
        history_features["dias_en_mercado"] = (history_features["last_date"] - history_features["first_date"]).dt.days.clip(lower=0)
        history_features["last_reprice_date"] = history_features["last_reprice_date"].fillna(history_features["last_date"])
        history_features["dias_desde_ultimo_repricing"] = (history_features["last_date"] - history_features["last_reprice_date"]).dt.days.clip(lower=0)
        history_features["descuento_desde_inicial_hist_pct"] = np.where(
            history_features["first_price"].notna() & (history_features["first_price"] > 0) & history_features["prev_price"].notna(),
            ((history_features["prev_price"] / history_features["first_price"]) - 1.0) * 100.0,
            np.nan,
        )
        history_features = history_features[[
            "listing_history_key", "dias_en_mercado", "republicaciones_count", "num_repricings",
            "descuento_desde_inicial_hist_pct", "dias_desde_ultimo_repricing",
            "n_observaciones_hist", "n_batches_hist", "n_urls_hist",
        ]]
        print(f"   Features históricas construidas: {len(history_features):,} llaves en {time.time() - history_t0:.1f}s")
    except Exception as e:
        print(f"⚠️ No se pudieron construir features históricas: {e}")

if not history_features.empty:
    df = df.merge(history_features, on="listing_history_key", how="left")
    history_feature_coverage = {col: float(df[col].notna().mean() * 100) for col in history_feature_cols}
else:
    for col in history_feature_cols:
        df[col] = np.nan

for col in ["republicaciones_count", "num_repricings", "n_observaciones_hist", "n_batches_hist", "n_urls_hist"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("\n🧾 Cobertura de features nuevas:")
for col in ["score_consenso_cross_portal", "dias_en_mercado", "num_repricings", "amenities_count"]:
    if col in history_feature_coverage:
        coverage = history_feature_coverage[col]
    else:
        coverage = float(df[col].notna().mean() * 100) if col in df.columns else 0.0
    print(f"   {col:32s}: {coverage:5.1f}%")

# =============================================================
# 3. FILTRAR OUTLIERS — precio_m2 por segmento mercado
# =============================================================
df = df[(df["area_m2"] >= 20) & (df["area_m2"] <= 800) & (df["precio_num"] > 0)].copy()
df["precio_m2"] = df["precio_num"] / df["area_m2"]
df["market_segment"] = df["city_token"].astype(str) + "__" + df["tipo_inmueble"].astype(str)
df["micro_market_segment"] = (
    df["city_token"].astype(str) + "__" + df["tipo_inmueble"].astype(str) + "__" + df["comuna_mercado"].astype(str)
)
df["log_area_m2"] = np.log1p(df["area_m2"])
df["banos_por_habitacion"] = (df["banos"] / df["habitaciones"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

df["dias_en_mercado"] = pd.to_numeric(df["dias_en_mercado"], errors="coerce").clip(lower=0, upper=3650)
df["dias_desde_ultimo_repricing"] = pd.to_numeric(df["dias_desde_ultimo_repricing"], errors="coerce").clip(lower=0, upper=3650)
df["descuento_desde_inicial_hist_pct"] = pd.to_numeric(df["descuento_desde_inicial_hist_pct"], errors="coerce").clip(lower=-80, upper=80)

outlier_t0 = time.time()
segment_bounds = (
    df.groupby("micro_market_segment")
    .agg(
        segment_n=("precio_m2", "size"),
        pm2_p10=("precio_m2", lambda s: s.quantile(0.10)),
        pm2_p90=("precio_m2", lambda s: s.quantile(0.90)),
    )
    .reset_index()
)
segment_bounds["pm2_iqr"] = segment_bounds["pm2_p90"] - segment_bounds["pm2_p10"]
segment_bounds["pm2_lower"] = (segment_bounds["pm2_p10"] - 1.5 * segment_bounds["pm2_iqr"]).clip(lower=0)
segment_bounds["pm2_upper"] = segment_bounds["pm2_p90"] + 1.5 * segment_bounds["pm2_iqr"]

df_filtered = df.merge(
    segment_bounds[["micro_market_segment", "segment_n", "pm2_lower", "pm2_upper"]],
    on="micro_market_segment",
    how="left"
)
df_filtered = df_filtered[
    (df_filtered["segment_n"] < 20) |
    (df_filtered["precio_m2"].between(df_filtered["pm2_lower"], df_filtered["pm2_upper"]))
]
global_p02 = df_filtered["precio_m2"].quantile(0.02)
global_p98 = df_filtered["precio_m2"].quantile(0.98)
df_clean = df_filtered[df_filtered["precio_m2"].between(global_p02, global_p98)].copy()
df_clean = df_clean.drop(columns=["segment_n", "pm2_lower", "pm2_upper"], errors="ignore")
print(f"🧹 Outlier filter (precio_m2 por microsegmento comuna): {len(df_clean):,} registros (de {len(df):,}) en {time.time() - outlier_t0:.1f}s")

print("\n" + "=" * 60)
print("  DIAGNÓSTICO DE DISTRIBUCIÓN DE PRECIOS")
print("=" * 60)
print(df_clean["precio_num"].describe(percentiles=[.05, .10, .25, .50, .75, .90, .95, .99]).to_string())
print(f"\n  Mediana:  ${df_clean['precio_num'].median() / 1e6:.0f}M")
print(f"  Media:    ${df_clean['precio_num'].mean() / 1e6:.0f}M")
print(f"  Skewness: {df_clean['precio_num'].skew():.2f}")
print(f"  precio_m2 mediano: ${df_clean['precio_m2'].median() / 1e6:.2f}M")

# =============================================================
# 4. PREPARACIÓN DE FEATURES LEAK-SAFE
# =============================================================
engineered_numeric_candidates = [
    "score_consenso_cross_portal",
    "dias_en_mercado",
    "republicaciones_count",
    "num_repricings",
    "descuento_desde_inicial_hist_pct",
    "dias_desde_ultimo_repricing",
    "n_observaciones_hist",
    "n_batches_hist",
    "n_urls_hist",
    "amenities_count",
    "amenity_lujo_score",
] + amenity_feature_cols

cols_numeric_base = [
    c for c in [
        "area_m2", "log_area_m2", "habitaciones", "banos", "garajes",
        "banos_por_habitacion", "num_portales", "dispersion_pct_grupo",
        "precio_desviacion_grupo_pct", "data_completeness",
    ] + engineered_numeric_candidates
    if c in df_clean.columns and df_clean[c].notna().mean() > 0.20
]
cols_categorical = [
    c for c in ["tipo_inmueble", "estado_inmueble", "fuente", "city_token", "comuna_mercado"]
    if c in df_clean.columns and df_clean[c].nunique() > 1
]

feature_cols_base = cols_numeric_base + cols_categorical + ["texto_completo", "market_segment", "micro_market_segment"]
X_raw = df_clean[feature_cols_base].copy()
y = df_clean["precio_num"].copy()

sample_weight = pd.Series(1.0, index=df_clean.index)
if "num_portales" in df_clean.columns:
    sample_weight += df_clean["num_portales"].fillna(1).clip(lower=1, upper=4).sub(1) * 0.20
if "dispersion_pct_grupo" in df_clean.columns:
    sample_weight *= np.where(df_clean["dispersion_pct_grupo"].fillna(0) <= 15, 1.10, 0.95)
    sample_weight *= np.where(df_clean["dispersion_pct_grupo"].fillna(0) >= 35, 0.85, 1.00)
if "score_consenso_cross_portal" in df_clean.columns:
    sample_weight *= np.where(df_clean["score_consenso_cross_portal"].fillna(50) >= 70, 1.08, 1.00)
    sample_weight *= np.where(df_clean["score_consenso_cross_portal"].fillna(50) <= 35, 0.92, 1.00)
sample_weight *= np.where(df_clean["comuna_mercado"].ne("comuna_otra"), 1.05, 0.95)
sample_weight = sample_weight.clip(lower=0.70, upper=1.80)

X_train_raw, X_test_raw, y_train, y_test, w_train, w_test = train_test_split(
    X_raw, y, sample_weight, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train_raw):,} | Test: {len(X_test_raw):,}")

def compute_market_features(train_x, train_y, apply_x):
    train_market = train_x[[
        "city_token",
        "comuna_mercado",
        "market_segment",
        "micro_market_segment",
        "fuente",
        "area_m2",
        "habitaciones",
    ]].copy()
    train_market["precio_num"] = train_y.values
    train_market = train_market.dropna(subset=[
        "city_token", "comuna_mercado", "market_segment", "micro_market_segment",
        "fuente", "area_m2", "precio_num"
    ])
    train_market = train_market[train_market["area_m2"] > 0].copy()
    train_market["precio_m2_train"] = train_market["precio_num"] / train_market["area_m2"]
    train_market["habitaciones_bucket"] = train_market["habitaciones"].fillna(-1).clip(-1, 6)

    pm2_p01 = train_market["precio_m2_train"].quantile(0.01)
    pm2_p99 = train_market["precio_m2_train"].quantile(0.99)
    train_market["precio_m2_train"] = train_market["precio_m2_train"].clip(pm2_p01, pm2_p99)

    city_stats = (
        train_market
        .groupby("city_token")
        .agg(
            precio_mediano_ciudad=("precio_num", "median"),
            precio_m2_mediano_ciudad=("precio_m2_train", "median"),
            area_mediana_ciudad=("area_m2", "median"),
        )
        .reset_index()
    )

    comuna_stats = (
        train_market
        .groupby(["city_token", "comuna_mercado"])
        .agg(
            precio_mediano_comuna=("precio_num", "median"),
            precio_m2_mediano_comuna=("precio_m2_train", "median"),
            comuna_n=("precio_num", "size"),
        )
        .reset_index()
    )

    segment_stats = (
        train_market
        .groupby("market_segment")
        .agg(
            precio_m2_mediano_segmento=("precio_m2_train", "median"),
            precio_mediano_segmento=("precio_num", "median"),
        )
        .reset_index()
    )

    micro_segment_stats = (
        train_market
        .groupby("micro_market_segment")
        .agg(
            precio_m2_mediano_microsegmento=("precio_m2_train", "median"),
            precio_mediano_microsegmento=("precio_num", "median"),
            micro_segmento_n=("precio_num", "size"),
        )
        .reset_index()
    )

    segment_profile_stats = (
        train_market
        .groupby("market_segment")
        .agg(
            segmento_n=("area_m2", "size"),
            area_mediana_segmento=("area_m2", "median"),
            area_p25_segmento=("area_m2", lambda s: s.quantile(0.25)),
            area_p75_segmento=("area_m2", lambda s: s.quantile(0.75)),
        )
        .reset_index()
    )

    hab_stats = (
        train_market
        .groupby(["city_token", "habitaciones_bucket"])
        .agg(precio_m2_mediano_habs=("precio_m2_train", "median"))
        .reset_index()
    )

    room_freq = (
        train_market
        .groupby(["market_segment", "habitaciones_bucket"])
        .size()
        .reset_index(name="habitaciones_segmento_n")
        .merge(segment_profile_stats[["market_segment", "segmento_n"]], on="market_segment", how="left")
    )
    room_freq["share_habitaciones_segmento"] = room_freq["habitaciones_segmento_n"] / room_freq["segmento_n"].replace(0, np.nan)

    source_stats = train_market.merge(segment_stats, on="market_segment", how="left")
    source_stats["ratio_vs_segment"] = source_stats["precio_num"] / (
        source_stats["area_m2"] * source_stats["precio_m2_mediano_segmento"].replace(0, np.nan)
    )
    source_stats["ratio_vs_segment"] = source_stats["ratio_vs_segment"].replace([np.inf, -np.inf], np.nan)
    source_stats = source_stats.dropna(subset=["ratio_vs_segment"])
    source_stats["ratio_vs_segment"] = source_stats["ratio_vs_segment"].clip(
        source_stats["ratio_vs_segment"].quantile(0.01),
        source_stats["ratio_vs_segment"].quantile(0.99),
    )

    fuente_ratio_stats = (
        source_stats.groupby("fuente")
        .agg(
            fuente_factor=("ratio_vs_segment", "median"),
            fuente_n=("ratio_vs_segment", "size"),
        )
        .reset_index()
    )

    fuente_segmento_ratio_stats = (
        source_stats.groupby(["fuente", "market_segment"])
        .agg(
            fuente_segmento_factor=("ratio_vs_segment", "median"),
            fuente_segmento_n=("ratio_vs_segment", "size"),
        )
        .reset_index()
    )

    result = apply_x.copy()
    result["habitaciones_bucket"] = result["habitaciones"].fillna(-1).clip(-1, 6)
    result = result.merge(city_stats, on="city_token", how="left")
    result = result.merge(comuna_stats, on=["city_token", "comuna_mercado"], how="left")
    result = result.merge(segment_stats, on="market_segment", how="left")
    result = result.merge(micro_segment_stats, on="micro_market_segment", how="left")
    result = result.merge(segment_profile_stats, on="market_segment", how="left")
    result = result.merge(hab_stats, on=["city_token", "habitaciones_bucket"], how="left")
    result = result.merge(room_freq[["market_segment", "habitaciones_bucket", "share_habitaciones_segmento"]], on=["market_segment", "habitaciones_bucket"], how="left")
    result = result.merge(fuente_ratio_stats, on="fuente", how="left")
    result = result.merge(fuente_segmento_ratio_stats, on=["fuente", "market_segment"], how="left")

    global_price_median = train_market["precio_num"].median()
    global_pm2_median = train_market["precio_m2_train"].median()
    global_area_median = train_market["area_m2"].median()
    global_fuente_factor = float(source_stats["ratio_vs_segment"].median()) if len(source_stats) else 1.0

    result["precio_mediano_ciudad"] = result["precio_mediano_ciudad"].fillna(global_price_median)
    result["precio_m2_mediano_ciudad"] = result["precio_m2_mediano_ciudad"].fillna(global_pm2_median)
    result["area_mediana_ciudad"] = result["area_mediana_ciudad"].fillna(global_area_median)

    result["precio_mediano_comuna"] = result["precio_mediano_comuna"].fillna(result["precio_mediano_ciudad"])
    result["precio_m2_mediano_comuna"] = result["precio_m2_mediano_comuna"].fillna(result["precio_m2_mediano_ciudad"])
    result["precio_m2_mediano_segmento"] = result["precio_m2_mediano_segmento"].fillna(result["precio_m2_mediano_comuna"])
    result["precio_mediano_segmento"] = result["precio_mediano_segmento"].fillna(result["precio_mediano_comuna"])

    result["precio_m2_mediano_microsegmento"] = result["precio_m2_mediano_microsegmento"].fillna(result["precio_m2_mediano_segmento"])
    result["precio_mediano_microsegmento"] = result["precio_mediano_microsegmento"].fillna(result["precio_mediano_segmento"])
    result["precio_m2_mediano_habs"] = result["precio_m2_mediano_habs"].fillna(result["precio_m2_mediano_microsegmento"])

    result["precio_m2_mediano_microsegmento"] = np.where(
        result["micro_segmento_n"].fillna(0) >= 12,
        result["precio_m2_mediano_microsegmento"],
        result["precio_m2_mediano_comuna"]
    )
    result["precio_mediano_microsegmento"] = np.where(
        result["micro_segmento_n"].fillna(0) >= 12,
        result["precio_mediano_microsegmento"],
        result["precio_mediano_comuna"]
    )

    result["precio_m2_mediano_comuna"] = np.where(
        result["comuna_n"].fillna(0) >= 15,
        result["precio_m2_mediano_comuna"],
        result["precio_m2_mediano_ciudad"]
    )
    result["precio_mediano_comuna"] = np.where(
        result["comuna_n"].fillna(0) >= 15,
        result["precio_mediano_comuna"],
        result["precio_mediano_ciudad"]
    )

    result["fuente_factor"] = result["fuente_factor"].fillna(global_fuente_factor)
    result["fuente_segmento_factor"] = result["fuente_segmento_factor"].fillna(result["fuente_factor"])
    result["fuente_segmento_factor"] = np.where(
        result["fuente_segmento_n"].fillna(0) >= 25,
        result["fuente_segmento_factor"],
        result["fuente_factor"]
    )

    result["precio_estimado_ciudad_area"] = result["area_m2"] * result["precio_m2_mediano_ciudad"]
    result["precio_estimado_comuna_area"] = result["area_m2"] * result["precio_m2_mediano_comuna"]
    result["precio_estimado_segmento_area"] = result["area_m2"] * result["precio_m2_mediano_segmento"]
    result["precio_estimado_microsegmento_area"] = result["area_m2"] * result["precio_m2_mediano_microsegmento"]
    result["precio_estimado_segmento_area_ajustado"] = result["precio_estimado_microsegmento_area"] * result["fuente_segmento_factor"]
    result["precio_estimado_segmento_area_ajustado"] = result["precio_estimado_segmento_area_ajustado"].fillna(result["precio_estimado_microsegmento_area"])
    result["precio_estimado_segmento_area_ajustado"] = result["precio_estimado_segmento_area_ajustado"].fillna(result["precio_estimado_comuna_area"])
    result["precio_estimado_segmento_area_ajustado"] = result["precio_estimado_segmento_area_ajustado"].fillna(result["precio_estimado_ciudad_area"])
    result["area_vs_ciudad_ratio"] = result["area_m2"] / result["area_mediana_ciudad"].replace(0, np.nan)
    result["area_vs_ciudad_ratio"] = result["area_vs_ciudad_ratio"].replace([np.inf, -np.inf], np.nan)
    result["ajuste_fuente_pct"] = (result["fuente_segmento_factor"] - 1.0) * 100.0

    result["area_mediana_segmento"] = result["area_mediana_segmento"].fillna(result["area_m2"].median())
    area_spread = (result["area_p75_segmento"] - result["area_p25_segmento"]).replace(0, np.nan)
    area_fallback_scale = result["area_mediana_segmento"].abs().replace(0, np.nan) * 0.25
    area_scale = area_spread.fillna(area_fallback_scale).fillna(max(float(train_market["area_m2"].median() * 0.25), 1.0))
    result["share_habitaciones_segmento"] = result["share_habitaciones_segmento"].fillna(0.10).clip(lower=0.01, upper=1.0)
    result["rareza_area_segmento"] = ((result["area_m2"] - result["area_mediana_segmento"]).abs() / area_scale).clip(lower=0.0, upper=8.0)
    result["rareza_habitaciones_segmento"] = (1.0 - result["share_habitaciones_segmento"]).clip(lower=0.0, upper=1.0)
    result["rareza_inmueble_segmento"] = (
        0.65 * result["rareza_area_segmento"] +
        0.35 * (result["rareza_habitaciones_segmento"] * 4.0)
    ).clip(lower=0.0, upper=8.0)
    result["rareza_segmento_inversa"] = (1.0 / np.sqrt(result["segmento_n"].fillna(1).clip(lower=1))).clip(lower=0.0, upper=1.0)

    return result, city_stats, comuna_stats, segment_stats, micro_segment_stats, fuente_ratio_stats, fuente_segmento_ratio_stats, {
        "global_price_median": global_price_median,
        "global_pm2_median": global_pm2_median,
        "global_area_median": global_area_median,
        "global_fuente_factor": global_fuente_factor,
        "pm2_clip_p01": pm2_p01,
        "pm2_clip_p99": pm2_p99,
    }

market_t0 = time.time()
print("\n🏙️ Calculando features de mercado (solo desde train)...")
X_train_enriched, city_stats, comuna_stats, segment_stats, micro_segment_stats, fuente_ratio_stats, fuente_segmento_ratio_stats, market_meta = compute_market_features(X_train_raw, y_train, X_train_raw)
X_test_enriched, _, _, _, _, _, _, _ = compute_market_features(X_train_raw, y_train, X_test_raw)
print(f"   Features de mercado listas en {time.time() - market_t0:.1f}s")

market_features = [
    "precio_mediano_ciudad",
    "precio_mediano_comuna",
    "precio_mediano_segmento",
    "precio_mediano_microsegmento",
    "precio_m2_mediano_ciudad",
    "precio_m2_mediano_comuna",
    "precio_m2_mediano_segmento",
    "precio_m2_mediano_microsegmento",
    "precio_m2_mediano_habs",
    "precio_estimado_ciudad_area",
    "precio_estimado_comuna_area",
    "precio_estimado_segmento_area",
    "precio_estimado_microsegmento_area",
    "precio_estimado_segmento_area_ajustado",
    "fuente_factor",
    "fuente_segmento_factor",
    "ajuste_fuente_pct",
    "area_vs_ciudad_ratio",
    "rareza_area_segmento",
    "rareza_habitaciones_segmento",
    "rareza_inmueble_segmento",
    "rareza_segmento_inversa",
]
cols_numeric_final = cols_numeric_base + market_features

for col in market_features:
    train_nan = int(X_train_enriched[col].isna().sum())
    test_nan = int(X_test_enriched[col].isna().sum())
    print(f"  {col:34s} train NaN={train_nan} | test NaN={test_nan}")

X_train_enriched["area_vs_ciudad_ratio"] = X_train_enriched["area_vs_ciudad_ratio"].fillna(1.0)
X_test_enriched["area_vs_ciudad_ratio"] = X_test_enriched["area_vs_ciudad_ratio"].fillna(1.0)

unseen_test_cities = int((~X_test_raw["city_token"].isin(X_train_raw["city_token"].unique())).sum())
print(f"  Ciudades no vistas en test: {unseen_test_cities}")
print(f"  Comunas de mercado train: {X_train_raw['comuna_mercado'].nunique():,}")
print(f"  Microsegmentos train: {X_train_raw['micro_market_segment'].nunique():,}")

X_train = X_train_enriched[cols_numeric_final + cols_categorical + ["texto_completo"]].copy()
X_test = X_test_enriched[cols_numeric_final + cols_categorical + ["texto_completo"]].copy()

def build_model():
    transformers = [
        (
            "txt",
            TfidfVectorizer(
                max_features=100,
                ngram_range=(1, 2),
                min_df=3,
                stop_words=[
                    "en", "de", "la", "el", "y", "con", "se", "del",
                    "por", "los", "las", "un", "una", "para", "al",
                ],
                sublinear_tf=True,
            ),
            "texto_completo",
        ),
        ("num", SimpleImputer(strategy="median"), cols_numeric_final),
    ]

    if cols_categorical:
        transformers.append(
            (
                "cat",
                Pipeline([
                    ("imp", SimpleImputer(strategy="constant", fill_value="desconocido")),
                    ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=5)),
                ]),
                cols_categorical,
            )
        )

    preprocessor = ColumnTransformer(transformers)
    xgb = XGBRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.70,
        min_child_weight=5,
        gamma=0.05,
        reg_alpha=0.20,
        reg_lambda=1.4,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    return Pipeline([
        ("preprocessor", preprocessor),
        (
            "regressor",
            TransformedTargetRegressor(
                regressor=xgb,
                func=np.log1p,
                inverse_func=np.expm1,
            ),
        ),
    ])

def evaluate_predictions(name, y_true, y_pred):
    return {
        "strategy": name,
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred) * 100),
    }

print("\n🚀 Entrenando modelo absoluto...")
t0 = time.time()
absolute_model = build_model()
absolute_model.fit(X_train, y_train, regressor__sample_weight=w_train.values)
y_pred_abs = absolute_model.predict(X_test)
elapsed_abs = time.time() - t0
metrics_abs = evaluate_predictions("absolute", y_test, y_pred_abs)

print("🚀 Entrenando modelo residual calibrado por fuente...")
baseline_train = X_train_enriched["precio_estimado_segmento_area_ajustado"].copy()
baseline_test = X_test_enriched["precio_estimado_segmento_area_ajustado"].copy()
baseline_train = baseline_train.fillna(X_train_enriched["precio_estimado_microsegmento_area"])
baseline_test = baseline_test.fillna(X_test_enriched["precio_estimado_microsegmento_area"])
baseline_train = baseline_train.fillna(X_train_enriched["precio_estimado_comuna_area"])
baseline_test = baseline_test.fillna(X_test_enriched["precio_estimado_comuna_area"])
baseline_train = baseline_train.fillna(X_train_enriched["precio_estimado_segmento_area"])
baseline_test = baseline_test.fillna(X_test_enriched["precio_estimado_segmento_area"])
baseline_train = baseline_train.fillna(X_train_enriched["precio_estimado_ciudad_area"])
baseline_test = baseline_test.fillna(X_test_enriched["precio_estimado_ciudad_area"])
baseline_train = baseline_train.fillna(float(y_train.median()))
baseline_test = baseline_test.fillna(float(y_train.median()))
baseline_train = baseline_train.clip(lower=50_000_000)
baseline_test = baseline_test.clip(lower=50_000_000)

residual_train_mask = y_train.notna() & baseline_train.notna() & np.isfinite(baseline_train) & (baseline_train > 0)
print(f"   Residual train válidos: {int(residual_train_mask.sum()):,}/{len(residual_train_mask):,}")
y_train_ratio = (y_train[residual_train_mask] / baseline_train[residual_train_mask]).clip(lower=0.20, upper=4.50)
X_train_res = X_train.loc[residual_train_mask].copy()
w_train_res = w_train.loc[residual_train_mask].copy()

t1 = time.time()
residual_model = build_model()
residual_model.fit(X_train_res, y_train_ratio, regressor__sample_weight=w_train_res.values)
pred_ratio_train = pd.Series(residual_model.predict(X_train_res))
ratio_clip_low = max(0.25, float(pred_ratio_train.quantile(0.01)))
ratio_clip_high = min(4.00, float(pred_ratio_train.quantile(0.99)))
pred_ratio_test = np.clip(residual_model.predict(X_test), ratio_clip_low, ratio_clip_high)
y_pred_res = pred_ratio_test * baseline_test.to_numpy()
elapsed_res = time.time() - t1
metrics_res = evaluate_predictions("residual", y_test, y_pred_res)

comparison = pd.DataFrame([metrics_abs, metrics_res]).sort_values(["mape", "mae"])
print("\n📊 COMPARACIÓN DE ESTRATEGIAS:")
print(comparison.to_string(index=False))

if metrics_res["mape"] < metrics_abs["mape"]:
    best_strategy = "residual"
    best_model = residual_model
    y_pred = y_pred_res
    r2 = metrics_res["r2"]
    mae = metrics_res["mae"]
    mape = metrics_res["mape"]
    elapsed = elapsed_res
else:
    best_strategy = "absolute"
    best_model = absolute_model
    y_pred = y_pred_abs
    r2 = metrics_abs["r2"]
    mae = metrics_abs["mae"]
    mape = metrics_abs["mape"]
    elapsed = elapsed_abs

print(f"\n🏆 Estrategia ganadora: {best_strategy}")
print(f"   R²:   {r2:.4f}")
print(f"   MAE:  ${mae:,.0f} COP")
print(f"   MAPE: {mape:.1f}%")
print(f"   Tiempo: {elapsed:.0f}s")
print("\n⏭️ Cross-validation omitido en esta versión: las features de mercado son fold-dependientes y hacerlo bien requiere recalcularlas dentro de cada fold.")

print("\n📊 MAPE por rango:")
df_eval = pd.DataFrame({"real": y_test.values, "pred": y_pred})
df_eval["rango"] = pd.cut(
    df_eval["real"],
    bins=[0, 300e6, 500e6, 800e6, 1500e6, float("inf")],
    labels=["<300M", "300-500M", "500-800M", "800M-1.5B", ">1.5B"],
)
mape_table = df_eval.groupby("rango", observed=True).apply(
    lambda x: pd.Series({
        "n": len(x),
        "mape_%": round((abs(x.pred - x.real) / x.real * 100).mean(), 1),
        "mae_M": round(abs(x.pred - x.real).mean() / 1e6, 0),
    }),
    include_groups=False,
 )
print(mape_table.to_string())

print("\n" + "=" * 60)
print("  DIAGNÓSTICO DE LEAKAGE")
print("=" * 60)
print("  No se usa precio_num/area_m2 como feature de entrada.")
print("  Las features de mercado se calculan solo desde train y luego se aplican a test.")
print("  `descuento_desde_inicial_hist_pct` usa precio previo observado, no el precio actual objetivo.")
print(f"  City stats calculados sobre {len(city_stats):,} ciudades/zonas colapsadas.")
print(f"  Comuna stats calculados sobre {len(comuna_stats):,} ciudad+comuna.")
print(f"  Segment stats calculados sobre {len(segment_stats):,} ciudad+tipo.")
print(f"  Microsegment stats calculados sobre {len(micro_segment_stats):,} ciudad+tipo+comuna.")
print(f"  Fuente stats calculados sobre {len(fuente_ratio_stats):,} fuentes.")
print(f"  Ciudades no vistas en test resueltas con fallback global: {unseen_test_cities}")
print("  Sample weights: más peso a inmuebles con mayor consenso cross-portal, menor dispersión y comunas conocidas.")
if best_strategy == "residual":
    print(f"  Ratio residual clip: [{ratio_clip_low:.2f}, {ratio_clip_high:.2f}]")
print("=" * 60)

try:
    xgb_step = best_model.named_steps["regressor"].regressor_
    feat_names = best_model.named_steps["preprocessor"].get_feature_names_out()
    imps = pd.Series(xgb_step.feature_importances_, index=feat_names)

    print("\n📊 Top 30 features por importancia:")
    print(imps.sort_values(ascending=False).head(30).to_string())

    groups = {
        "texto_tfidf": imps[imps.index.str.startswith("txt__")].sum(),
        "numericas_total": imps[imps.index.str.startswith("num__")].sum(),
        "categoricas_total": imps[imps.index.str.startswith("cat__")].sum(),
        "num__precio_estimado_microsegmento_area": imps[imps.index.str.contains("num__precio_estimado_microsegmento_area", regex=False)].sum(),
        "num__precio_estimado_segmento_area_ajustado": imps[imps.index.str.contains("num__precio_estimado_segmento_area_ajustado", regex=False)].sum(),
        "num__score_consenso_cross_portal": imps[imps.index.str.contains("num__score_consenso_cross_portal", regex=False)].sum(),
        "num__dias_en_mercado": imps[imps.index.str.contains("num__dias_en_mercado", regex=False)].sum(),
        "num__num_repricings": imps[imps.index.str.contains("num__num_repricings", regex=False)].sum(),
        "num__amenities_count": imps[imps.index.str.contains("num__amenities_count", regex=False)].sum(),
        "num__rareza_inmueble_segmento": imps[imps.index.str.contains("num__rareza_inmueble_segmento", regex=False)].sum(),
        "cat__comuna_mercado": imps[imps.index.str.contains("cat__comuna_mercado", regex=False)].sum(),
        "cat__city_token": imps[imps.index.str.contains("cat__city_token", regex=False)].sum(),
    }

    print("\n📊 Importancia por grupo:")
    for name, imp in sorted(groups.items(), key=lambda item: -item[1]):
        print(f"   {name:40s}: {imp:.4f}  ({imp * 100:.1f}%)")
except Exception as e:
    print(f"No se pudo extraer importancias: {e}")

model_bundle = {
    "model": best_model,
    "strategy": best_strategy,
    "city_stats": city_stats,
    "comuna_stats": comuna_stats,
    "segment_stats": segment_stats,
    "micro_segment_stats": micro_segment_stats,
    "fuente_ratio_stats": fuente_ratio_stats,
    "fuente_segmento_ratio_stats": fuente_segmento_ratio_stats,
    "market_meta": market_meta,
    "feature_cols": cols_numeric_final + cols_categorical + ["texto_completo"],
    "market_features": market_features,
    "history_feature_cols": history_feature_cols,
    "amenity_feature_cols": amenity_feature_cols,
    "ratio_clip_low": ratio_clip_low if best_strategy == "residual" else None,
    "ratio_clip_high": ratio_clip_high if best_strategy == "residual" else None,
    "trained_at": datetime.now(tz=timezone.utc).isoformat(),
    "metrics": {
        "r2": r2,
        "mae": mae,
        "mape": mape,
        "train_size": len(X_train),
        "test_size": len(X_test),
    },
    "comparison": comparison.to_dict(orient="records"),
    "model_type": "bundle_v5_historia_consenso",
}

s3 = boto3.client(
    "s3",
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"],
)

ts = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
model_key = f"models/modelo_xgboost_bundle_v5_historia_consenso_{ts}.pkl"

buf = BytesIO()
pickle.dump(model_bundle, buf)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, model_key)
print(f"\n💾 Bundle challenger guardado: s3://{BUCKET}/{model_key}")

manifest_key = "models/manifest.json"
previous_manifest = {}
previous_manifest_exists = False
try:
    previous_manifest = json.loads(s3.get_object(Bucket=BUCKET, Key=manifest_key)["Body"].read())
    previous_manifest_exists = True
    prev_mape = previous_manifest.get("metrics", {}).get("mape", float("inf"))
except Exception:
    prev_mape = float("inf")

should_promote = FORCE_PROMOTE_TO_CHAMPION or (AUTO_PROMOTE_IF_BETTER and mape < prev_mape) or (not previous_manifest_exists)
if should_promote:
    manifest_payload = {
        "champion_model_key": model_key,
        "model_type": model_bundle["model_type"],
        "strategy": best_strategy,
        "trained_at": model_bundle["trained_at"],
        "promoted_at": datetime.now(tz=timezone.utc).isoformat(),
        "metrics": model_bundle["metrics"],
        "comparison": model_bundle["comparison"],
        "promotion_reason": "forced" if FORCE_PROMOTE_TO_CHAMPION else ("better_mape" if previous_manifest_exists else "bootstrap"),
    }
    s3.put_object(
        Bucket=BUCKET,
        Key=manifest_key,
        Body=json.dumps(manifest_payload, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"🏆 Manifest actualizado: s3://{BUCKET}/{manifest_key}")
    print(f"   Nuevo principal: {model_key}")
else:
    print(f"📌 Campeón actual: {previous_manifest.get('champion_model_key')} | MAPE {prev_mape:.2f}%")
    print(f"📌 Challenger {best_strategy}/bundle_v5_historia_consenso: MAPE {mape:.2f}% | mejora {prev_mape - mape:+.2f} pp")
    print("⚠️ No se promovió a principal porque no mejora al campeón. Usa FORCE_PROMOTE_TO_CHAMPION=True solo si quieres forzarlo.")